# Gen AI Evaluation Service

Agent Platform の Gen AI 評価サービス（Gen AI Evaluation Service）を使用すると、特定の要件に従ってあらゆる Generative AI モデルやアプリケーションを評価できます。

一般的なリーダーボードやレポートのみに頼るのではなく、独自の評価基準を定義し、独自のニーズ、ユースケース、データに対してさまざまなモデルがどのように機能するかを直接比較できます。

このサービスにより、以下のことが可能になります。

- モデルのパフォーマンスをより深く理解する。一般的な指標を超えて、モデルが特定のデータやタスクをどのように処理するかを理解できます。
- 開発ライフサイクル全体で十分な情報に基づいた意思決定を行う。評価を活用してモデルの選択を導き、プロンプト エンジニアリングを改善し、モデルのカスタマイズを最適化します。
- Agent Platform の統合ツールを活用して、必要に応じて評価を簡単に開始および再利用することで、評価ワークフローを効率化する。

基本的に、Generative AI の評価を自身でコントロールできるようになり、選択したモデルが特定のアプリケーションに最適であることを確認できます。

## 学習目標

このノートブックでは、以下の内容について学びます。

- Gen AI Evaluation Service の使用方法
- さまざまな種類の評価手法（計算ベースとモデルベース）
- データセットを準備して評価に使える状態にする方法
- 評価結果を分析して理解する方法

### セットアップ

In [ ]:
# 全般
import os
import random
import string

import pandas as pd
from google import genai
from IPython.display import HTML, Markdown, display

# Agent Platform Evaluation（評価）
from vertexai.evaluation import (
    EvalTask,
    MetricPromptTemplateExamples,
    PairwiseMetric,
    PointwiseMetric,
)


### ヘルパー関数

この関数は、評価 SDK からの結果を読みやすい形式で表示するのに役立ちます。

In [ ]:
def display_eval_report(eval_result, metrics=None):
    """評価タスクの評価結果を表示します。"""
    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # For any nested dictionaries, display formatted JSON
    report_df = report_df.map(
        lambda x: (
            json.dumps(x, indent=2)
            if isinstance(x, dict)
            else (
                x.to_json(indent=2)
                if isinstance(x, (pd.Series, pd.DataFrame))
                else x
            )
        )
    )
    display(HTML(f"<h2>{title}</h2>"))
    display(
        HTML("<h3>Summary Metrics</h3>"),
        HTML(metrics_df.to_html(index=False)),
        HTML("<h3>Report Metrics</h3>"),
        HTML(report_df.to_html(index=False)),
    )


## 評価プロセス

Agent Platform の Gen AI 評価サービスを使用すると、特定のニーズや基準に基づいて任意の生成モデルを評価できます。

評価を支援するために従う 4 つのステップは以下のとおりです。

1. 定義（Define）: ビジネス目標に合わせてメトリクスを調整し、評価アプローチ（計算ベース vs モデルベース）を選択します。
2. 準備（Prepare）: 実際のユースケースを反映したデータセットを作成します。
3. 実行（Run）: テンプレートや既存のサンプルを使用して簡単に評価を開始します。モデルを定義し、他の評価で使用できるように Agent Platform 内で再利用可能な `EvalTasks` を作成します。
4. 分析（Analyze）: 結果を解釈し、各モデルが特定の基準に対してどのように機能するかを理解します。

これらのステップを 1 つずつ見ていきましょう。

### 計算ベースのメトリクス（Computation Based Metrics）

[計算ベースのメトリクス](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) は、数式を使用してモデルの出力を**グラウンド トゥルース（正解）またはリファレンス**と比較することによって計算されます。一般的に使用される計算ベースのメトリクスには、ROUGE や BLEU があります。よく使用されるメトリクスは、次のグループに分類できます。

- 語彙ベースのメトリクス: `Exact Match` や `ROUGE` など、数式を使用して LLM が生成した結果とグラウンド トゥルースの間の文字列類似度を計算します。
- カウントベースのメトリクス: `F1-score`、`Accuracy`、`Tool Name Match` など、特定のグラウンド トゥルース ラベルにヒットしたかミスしたかの行数を集計します。
- 埋め込みベースのメトリクス: 埋め込み空間における LLM 生成結果とグラウンド トゥルースとの間の距離を計算し、類似度レベルを反映します。これらには `cosine similarity`（コサイン類似度）や `euclidean distance`（ユークリッド距離）が含まれます。

ここでは、Gemini を使用した要約において、異なるプロンプトがどれだけ効果的に機能するかを評価してみましょう。まず `context` にいくつかの記事を定義します。評価に計算ベースのメトリクスを使用するため、要約のグラウンド トゥルースも定義する必要があります。これは `reference` で定義されます。`eval_dataset` は、評価に必要な列を含む DataFrame である必要があります。

**演習** `context`、`reference`、`instruction` 列を持つ `eval_dataset` DataFrame を作成してください。

In [ ]:
instruction = "Summarize the following article"

context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "To bake a decadent chocolate cake from scratch, start by preheating your oven to 350°F (175°C) and greasing and flouring two 9-inch round cake pans. In a large mixing bowl, cream together softened butter and granulated sugar until light and fluffy. Beat in eggs one at a time, making sure each egg is fully incorporated before adding the next. In a separate bowl, sift together all-purpose flour, cocoa powder, baking powder, baking soda, and salt. Divide the batter evenly between the prepared cake pans and bake for 25-30 minutes, or until a toothpick inserted into the center comes out clean.",
]

reference = [
    "The process of making spaghetti carbonara involves boiling pasta, crisping pancetta or guanciale, whisking together eggs and Parmesan cheese, and tossing everything together to create a creamy sauce.",
    "Preparing risotto entails sautéing onions and garlic, toasting Arborio rice, adding wine and broth gradually, and stirring until creamy and tender.",
    "Baking a decadent chocolate cake requires creaming butter and sugar, beating in eggs and alternating dry ingredients with buttermilk before baking until done.",
]

eval_dataset = None  # TODO create a pandas dataframe here

次に、評価するさまざまなプロンプト テンプレートを定義します。このリストには、試したいさまざまなプロンプト テンプレートをすべて含めることができます。デモンストレーションの目的で、以下では 2 つの異なるプロンプト テンプレートを使用します。

In [ ]:
prompt_templates = [
    "Instruction: {instruction}. Article: {context}. Summary:",
    "Article: {context}. Complete this task: {instruction}, in one sentence. Summary:",
]

次に、このタスクで測定するさまざまなメトリクスを定義します。タスクに基づいて Agent Platform で定義されているさまざまなメトリクスがあります。[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) を参照してさまざまなメトリクスを確認できます。また、それぞれの正しい文字列値については [API ドキュメント](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) を参照してください。

In [ ]:
metrics = [
    "rouge_1",
    "rouge_l_sum",
    "bleu",
    "safety",
]

モデル名（例: `'gemini-3.8-flash'`）を受け取り、指定されたモデルを呼び出してレスポンスを返す関数を定義しましょう。

In [ ]:
# 標準の Google GenAI クライアントを初期化
client = genai.Client(vertexai=True, location="global")


def create_model_fn(model_name: str):
    def model_fn(prompt: str) -> str:
        response = client.models.generate_content(
            model=model_name, contents=prompt
        )
        return response.text

    return model_fn


model_func_3_8_flash = create_model_fn("gemini-3.8-flash")


すべてのパラメータを設定できたので、評価を開始する準備が整いました。[EvalTask](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) を使用して、評価用の Agent Platform Experiment（実験）を開始します。

#### EvalTask クラスの理解

EvalTask クラスは、Gen AI Evaluation Service SDK フレームワークのコア コンポーネントです。Gen AI モデル / アプリケーションに対して評価ジョブを定義して実行でき、特定のタスクでのパフォーマンスを測定するための構造化された方法を提供します。EvalTask は評価プロセスの設計図と考えてください。評価タスクには、評価データセットと評価するメトリクスのリストが含まれている必要があります。サポートされているメトリクスは、Generative AI の[評価メトリクスの定義ページ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) に記載されています。データセットは `pandas.DataFrame`、Python 辞書、またはファイルパスの URI を指定でき、`prompt`、`reference`、`response`、`baseline_model_response` などのデフォルトの列名を含めることができます。

- Bring-your-own-response (BYOR): 評価するデータがすでにデータセットに保存されています。`response_column_name` や `baseline_model_response_column_name` などのパラメータを使用するか、`metric_column_mapping` を介して、使用するモデルとベースライン モデルの両方のレスポンス列名をカスタマイズできます。

- プロンプト テンプレートを使用しないモデル推論の実行: モデルへの入力プロンプトを含むデータセットがあり、評価前にモデル推論を実行したい場合です。評価データセットには `prompt` という名前の列が必要であり、これがモデルへの入力として直接使用されます。

- プロンプト テンプレートを使用したモデル推論の実行: プロンプト テンプレートへの入力変数を含むデータセットがあり、モデル推論用のプロンプトを組み立てたい場合です。評価データセットには、プロンプト テンプレート内の変数名に対応する列名が含まれている必要があります。たとえば、プロンプト テンプレートが "Instruction: {instruction}, context: {context}" の場合、データセットには `instruction` 列と `context` 列が含まれている必要があります。

EvalTask は、BYOR、Gemini モデルを使用したモデル推論、サードパーティ モデルのエンドポイント / SDK クライアント、カスタム モデル生成関数など、計算ベースのメトリクス、モデルベースのポイントワイズおよびペアワイズ メトリクスを使用した広範な評価シナリオをサポートしています。`evaluate()` メソッドは評価プロセスをトリガーし、オプションでモデル、プロンプト テンプレート、実験ログ構成、その他の評価実行構成を受け取ります。詳細については、[Gen AI Evaluation](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation) パッケージの SDK リファレンス ドキュメントを参照してください。

まず、プロンプト テンプレートを使用したモデル推論を使用します。

**演習** dataset、metrics、experiment パラメータを指定して `EvalTask` を定義してください。[EvalTask のドキュメント](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) を参照してください。

In [ ]:
experiment_name = "prompt-engineering-eval"

summarization_eval_task = None  # TODO: ここに EvalTask を定義


注: 以下のセルは実行が完了するまでに 10〜15 分かかります。

In [ ]:
def generate_uuid(length: int = 8) -> str:
    """指定された長さ（デフォルト=8）の UUID を生成します。"""
    return "".join(
        random.choices(
            string.ascii_lowercase + string.digits,
            k=length,
        )
    )


eval_results = []
for i, prompt_template in enumerate(prompt_templates):
    experiment_run_name = f"eval-{generate_uuid()}"
    eval_result = summarization_eval_task.evaluate(
        model=model_func_3_8_flash,
        prompt_template=prompt_template,
        experiment_run_name=experiment_run_name,
    )
    eval_results.append(
        (f"Prompt #{i}", eval_result.summary_metrics, eval_result.metrics_table)
    )


In [ ]:
for eval_result in eval_results:
    display_eval_report(eval_result)

以下の関数を使用して、実験実行の結果を表示できます。

In [ ]:
summarization_eval_task.display_runs()

### モデルベースのメトリクス（Model Based Metrics）

ユースケースのリファレンス結果やグラウンド トゥルースがない場合、別の LLM を使用してさまざまなモデルを評価および比較できます。この手法は**モデルベース評価（Model based evaluation）**と呼ばれます。モデルベースのメトリクスは、審査員（judge）モデルに対して候補モデルを評価します。ほとんどのユースケースにおける審査員モデルは Gemini ですが、翻訳のユースケースでは [MetricX](https://github.com/google-research/metricx) や [COMET](https://huggingface.co/Unbabel/wmt22-comet-da) などのモデルを使用することもできます。

モデルベースのメトリクスは、ペアワイズ（pairwise）またはポイントワイズ（pointwise）で測定できます。

- **ポイントワイズ メトリクス（Pointwise metrics）**: 審査員モデルに、評価基準に基づいて候補モデルの出力を評価させます。たとえば、スコアは 0 から 5 で、0 は回答が基準に合致していないことを意味し、5 は回答が基準によく合致していることを意味します。

- **ペアワイズ メトリクス（Pairwise metrics）**: 審査員モデルに 2 つのモデルの回答を比較させ、より優れた方を選択させます。これは、候補モデルとベースライン モデルを比較するときによく使用されます。ペアワイズ メトリクスは、Gemini を審査員モデルとして使用する場合にのみサポートされます。

#### ポイントワイズ メトリクス（Pointwise Metrics）

プロンプトを決定したところで、「流暢さ（fluency）」や「面白さ（entertaining）」などの 2 つの定性的基準で生成された要約も評価したいとします。これら 2 つの基準を使用して、`text_quality` というメトリクスを定義します。

In [ ]:
# text_quality の独自定義。
metric_prompt_template = PointwiseMetricPromptTemplate(
    criteria={
        "fluency": (
            "The summary should be free of grammatical errors, and have a good"
            " flow."
        ),
        "entertaining": (
            "The summary should be written in a tone that is engaging to read"
        ),
    },
    rating_rubric={
        "-1": (
            "The response does not meet both criteria of being fluent and"
            " entertaining."
        ),
        "0": (
            "The response partially meets one of the criteria of being fluent"
            " or entertaining."
        ),
        "1": (
            "The response completely meets both criteria of being fluent and"
            " entertaining."
        ),
    },
)

text_quality = PointwiseMetric(
    metric="text_quality",
    metric_prompt_template=metric_prompt_template,
)


この `text_quality` メトリクスがどのようなものか見てみましょう。

In [ ]:
print(text_quality.metric_prompt_template)

まず、対象のモデル（`gemini-3.8-flash`）からレスポンスを取得し、リストに保存する必要があります。

In [ ]:
context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "For a flavorful grilled steak, start by choosing a well-marbled cut of beef like ribeye or New York strip. Season the steak generously with kosher salt and freshly ground black pepper on both sides, pressing the seasoning into the meat. Preheat a grill to high heat and brush the grates with oil to prevent sticking. Place the seasoned steak on the grill and cook for about 4-5 minutes on each side for medium-rare, or adjust the cooking time to your desired level of doneness. Let the steak rest for a few minutes before slicing against the grain and serving.",
]

**演習:** 上記の `context` リストに対して、先ほど作成した `model_func_3_8_flash` を使用して Gemini から要約レスポンスを生成してください。

In [ ]:
responses = []
instruction = "Summarize the following article"
for article in context:
    prompt = None  # TODO: 上記の instruction と context を受け取るプロンプトをここに定義
    response = None  # TODO: model_func_3_8_flash() を使用して要約を取得
    responses.append(response)


**演習**: 評価用のデータセット `eval_dataset` を pandas DataFrame として定義してください。この `eval_dataset` には、上記で生成したレスポンスの 1 列のみが含まれます。

In [ ]:
eval_dataset = None  # TODO: 1 列の DataFrame を定義


In [ ]:
EXPERIMENT_NAME = "pointwise-eval"
eval_task = EvalTask(
    dataset=eval_dataset, metrics=[text_quality], experiment=EXPERIMENT_NAME
)

pointwise_eval_results = eval_task.evaluate()

ここでは、3 つの記事すべてに対する `summary_metrics` を確認できます。

In [ ]:
pointwise_eval_results.summary_metrics

上記で定義した基準に基づくと、レスポンスが両方の基準にある程度一致しているため、`text_quality` が `0.0` になっていることがわかります。`text_quality` で定義した基準に基づいた各記事のスコアを取得するには、`metrics_table` にアクセスします。

In [ ]:
pointwise_eval_results.metrics_table

定義した基準に基づくと、モデルは各要約に対してニュートラル（中立）なスコアを与えたようです。`text_quality/explanation` 列で、モデルのスコアの背後にある根拠を確認できます。

#### ペアワイズ メトリクス - モデルを並べて比較（SxS）

プロンプトを決定したところで、2 つの異なる LLM（Gemini 3.8 Flash vs Gemini 3.1 Pro）から生成された要約も評価したいとします。ペアワイズ モデル評価と並行（Side-by-Side: SxS）比較を使用して、2 つの異なるモデルからの要約を評価できます。

2 つのモデルを直接比較するには、`EvalTask` 実行内で `PairwiseMetric` を定義します。このアプローチにより、モデルのパフォーマンスを直接対決（head-to-head）で評価できます。

In [ ]:
instruction = "Summarize the following article"
prompt_template = "{instruction}. Article: {context}. Summary:"

pairwise_eval_dataset = pd.DataFrame(
    {
        "context": context,
        "instruction": [instruction] * len(context),
        "reference": reference,
    }
)

評価データセットを準備したら、比較する 2 つのモデルを定義します。`model_a` は `gemini-3.1-pro-preview`、`model_b` は `gemini-3.8-flash` になります。

In [ ]:
model_a = create_model_fn("gemini-3.1-pro-preview")
model_b = create_model_fn("gemini-3.8-flash")

上記のポイントワイズ評価の評価基準を定義した方法と同様に、評価プロンプトをカスタマイズできます。ただし、時間を節約するために、Agent Platform は使用可能な [事前定義された評価プロンプト](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/metrics-templates) を `MetricPromptTemplateExamples` で提供しています。このユースケースでは、2 つのモデル間のテキスト品質を評価します。`pairwise_text_quality` は、次の基準を使用してモデルのレスポンスを評価します。

```
STEP 1: Analyze Response A based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion.
STEP 2: Analyze Response B based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion 
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment of each criterion 
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field, justifying your choice by highlighting the specific strengths and weaknesses of each response in terms of Text Quality
```

In [ ]:
# 「Pairwise Text Quality」メトリクスを作成
text_quality_prompt_template = (
    MetricPromptTemplateExamples.get_prompt_template(
        "pairwise_text_quality"
    )
)

pairwise_text_quality_metric = PairwiseMetric(
    metric="pairwise_text_quality",
    metric_prompt_template=text_quality_prompt_template,
)


データセットと評価基準を定義したら、評価ジョブを開始する準備が整いました。

**演習** ここで `EvalTask` 関数を定義してください。ペアワイズ評価に必要なパラメータは、dataset、metrics、experiment です。

注: 以下のセルは実行が完了するまでに 10〜15 分かかります。

In [ ]:
pairwise_text_quality_eval_task = (
    None  # TODO: 適切なパラメータを指定して eval_task 関数をここに定義
)

# ペアワイズ比較の候補モデルを指定
pairwise_text_quality_eval_result = (
    pairwise_text_quality_eval_task.evaluate(
        model=model_a,
        baseline_model=model_b,
        experiment_run_name=f"pairwise-eval-{generate_uuid()}",
    )
)


上記の実験が実行されたら、`display_eval_report` ヘルパー関数を使用して結果を表示できます。

In [ ]:
display_eval_report(
    (
        "Side-by-side EvalTask",
        pairwise_text_quality_result.summary_metrics,
        pairwise_text_quality_result.metrics_table,
    )
)

ノートブックで実行した上記の評価実験はすべて、Agent Platform UI の [Experiments タブ](https://console.cloud.google.com/agent-platform/experiments/experiments) からアクセスできます。

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.